# Native Propagation Backends

For users familiar with QuTiP states and Hamiltonians. Install `pysuqu`, NumPy, Matplotlib, and QuTiP 5 in this notebook's Python environment. The optional C++ extension enables the native comparisons; every section also runs with QuTiP alone.

This tutorial covers sampled IQ controls, backend agreement, repeated initial states, and static-collapse evolution. All model parameters are synthetic and reproducible.

1. Build a sampled control and compare exact backends.
2. Reuse a context to form a unitary.
3. Check dissipative evolution against an analytic result.
4. Change the control amplitude and inspect conservation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import qutip as qt
from pysuqu.funclib.transmission import SignalTrace
from pysuqu.qubit import DriveTerm, PreparedPropagation, native_backend_available

rng = np.random.default_rng(764893)
backends = ['qutip', 'qutip_compiled']
if native_backend_available():
    backends.append('cpp')
print('Available comparisons:', ', '.join(backends))


## Sampled IQ And An Analytic Carrier

Time is measured in arbitrary consistent units. Hamiltonians use angular-frequency units, while `lo_freq` is in cycles per time unit. The solver reconstructs the real RF drive as I times cosine minus Q times sine. The default linear-IQ path retains the analytic carrier between sample times.


In [ ]:
times = np.linspace(0.0, 4.0, 161)
amplitude = float(rng.uniform(0.16, 0.24))
carrier = float(rng.uniform(0.65, 0.85))
phase = float(rng.uniform(-0.3, 0.3))
envelope = amplitude * np.sin(np.pi * times / times[-1]) ** 2 * np.exp(1j * phase)
trace = SignalTrace(times, envelope, 40.0, 'iq_complex', 'qubit_iq', carrier)
hamiltonian = 0.5 * 2 * np.pi * carrier * qt.sigmaz()
terms = [DriveTerm(qt.sigmax(), trace)]
options = {'atol': 1e-11, 'rtol': 1e-9, 'nsteps': 100000, 'coefficient_order': 1}
initial = qt.basis(2, 0)
prepared = {name: PreparedPropagation(hamiltonian, terms, times, backend=name, options=options)
            for name in backends}
results = {name: context.propagate(initial) for name, context in prepared.items()}
reference = results['qutip'].final_state
errors = {name: (result.final_state - reference).norm() for name, result in results.items()}
assert max(errors.values()) < 2e-6
print('Final-state errors:', errors)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), constrained_layout=True)
axes[0].plot(times, envelope.real, label='I')
axes[0].plot(times, envelope.imag, label='Q')
axes[0].set(xlabel='Time', ylabel='Drive amplitude')
axes[0].legend()
excited = qt.ket2dm(qt.basis(2, 1))
for name, result in results.items():
    axes[1].plot(result.times, qt.expect(excited, result.states), label=name,
                 linestyle='--' if name == 'cpp' else '-')
axes[1].set(xlabel='Time', ylabel='Excited-state population')
axes[1].legend()
plt.show()


## Reuse Preparation For A State Batch

Propagate both basis states with the same context to construct the unitary column by column. The native adapter can share preparation and schedule independent states in parallel. Check the Gram matrix to retain relative phases as well as state norms.


In [ ]:
selected = 'cpp' if 'cpp' in backends else 'qutip_compiled'
batch = prepared[selected].propagate_batch([qt.basis(2, 0), qt.basis(2, 1)])
unitary = np.column_stack([state.full()[:, 0] for state in batch.final_states])
unitarity_error = np.linalg.norm(unitary.conj().T @ unitary - np.eye(2))
assert unitarity_error < 2e-6
print('Backend:', selected, '| unitarity error:', unitarity_error)


## Static-Collapse Lindblad Evolution

A static relaxation operator converts the initial ket to a density matrix. With a zero Hamiltonian the excited-state population follows an exponential, providing an independent check. The native backend applies the master equation directly without building a Kronecker Liouvillian.


In [ ]:
decay_rate = float(rng.uniform(0.12, 0.22))
collapse = [np.sqrt(decay_rate) * qt.destroy(2)]
open_context = PreparedPropagation(qt.qzero(2), [], times, c_ops=collapse,
                                   backend=selected, options=options)
decay = open_context.propagate(qt.basis(2, 1))
population = np.asarray(qt.expect(excited, decay.states))
expected = np.exp(-decay_rate * times)
assert np.max(np.abs(population - expected)) < 2e-7
assert max(abs(state.tr() - 1.0) for state in decay.states) < 2e-7
fig, ax = plt.subplots(figsize=(6, 3.2), constrained_layout=True)
ax.plot(times, expected, label='Analytic decay')
ax.plot(times[::8], population[::8], 'o', label=selected)
ax.set(xlabel='Time', ylabel='Excited-state population')
ax.legend()
plt.show()


## Exercise: Change The Control Amplitude

Predict how reducing the envelope amplitude affects the transition probability. Complete the scaffold by trying scale factors between 0 and 1; rebuild the context for each changed trace. The no-drive result should retain the initial population. Every closed-system result should preserve norm.


In [ ]:
def final_population(scale):
    changed_trace = SignalTrace(times, scale * envelope, 40.0, 'iq_complex', 'qubit_iq', carrier)
    context = PreparedPropagation(hamiltonian, [DriveTerm(qt.sigmax(), changed_trace)], times,
                                  backend=selected, options={**options, 'store_states': False})
    state = context.propagate(initial).final_state
    assert abs(state.norm() - 1.0) < 2e-6
    return float(qt.expect(excited, state))

scales = [0.0, 0.5, 1.0]
populations = [final_population(scale) for scale in scales]
assert abs(populations[0]) < 1e-8
list(zip(scales, populations))


## Comparison Pitfalls

- Keep Hilbert-space dimension, trace grids, interpolation order, tolerances, and stored outputs identical across backends.
- `cpp_fast` applies a rotating-wave approximation and must be compared separately. `auto` does not select it.
- `cpp` raises an explicit error when the extension or requested capability is unavailable. `auto` can use compiled QuTiP for time-dependent collapse operators or `e_ops`.
- Prepare a new context when changing the Hamiltonian or trace; do not mutate an already prepared model.
- Run `python -m benchmarks.propagation_workflow --case gate` from a checkout for separate cold and warm timings. This notebook checks numerical behavior without making a speedup claim.

See the [backend guide](../docs/guides/propagation-backends.md) for installation, representation controls, cache settings, and benchmark variants.
